In [ ]:
import os
os.environ["TORCH_DYNAMO_DISABLE"] = "1"                          
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"                                             

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
import multiprocessing


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# =================================================================
                                
# =================================================================
class FourierFeatureMapping(nn.Module):
    def __init__(self, in_dims=2, mapping_size=128, scale=10.0):
        super().__init__()
                                  
        self.register_buffer('B', torch.randn(in_dims, mapping_size) * scale)

    def forward(self, x):
        # x: (Batch, 2)
        # x @ B: (Batch, mapping_size)
        x_proj = (2.0 * np.pi * x) @ self.B
                                          
        return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)

# =================================================================
               
# =================================================================
class DenseResBlock(nn.Module):
    def __init__(self, size):
        super().__init__()
        self.fc1 = nn.Linear(size, size)
        self.fc2 = nn.Linear(size, size)
        self.norm1 = nn.LayerNorm(size)
        self.norm2 = nn.LayerNorm(size)

    def forward(self, x):
        res = x
        x = self.norm1(x)
        x = F.silu(self.fc1(x))
        x = self.norm2(x)
        x = F.silu(self.fc2(x))
        return x + res

# =================================================================
                       
# =================================================================
class HighFidelityNeuralVF(nn.Module):
    def __init__(self, n_poles=8, width=256, fourier_size=128,num=6):
        super().__init__()
        self.n_poles = n_poles
        
                                         
        self.fourier_map = FourierFeatureMapping(in_dims=2, mapping_size=fourier_size, scale=5.0)
        
                                 
        in_features = fourier_size * 2 + 2 
        self.in_layer = nn.Linear(in_features, width)
        
                           
        self.res_layers = nn.ModuleList([DenseResBlock(width) for _ in range(num)])
        
                
        self.out_layer = nn.Linear(width, 2 * n_poles)

    def forward(self, x):
        # x shape: (Batch, 2)
        
                
        x_fourier = self.fourier_map(x)
                                            
        x_concat = torch.cat([x, x_fourier], dim=-1)
        
        h = F.silu(self.in_layer(x_concat))
        for layer in self.res_layers:
            h = layer(h)
        out = self.out_layer(h)
        
                             
        log_tau_k = out[:, 0 : self.n_poles]
        log_w_k = out[:, self.n_poles :]
        
        tau_k = 10.0 ** log_tau_k
        w_k = 10.0 ** log_w_k
                                  
                                        
        w_k_sum = torch.sum(w_k, dim=1, keepdim=True) + 1e-12        
        w_k = w_k / w_k_sum
        
        return tau_k, w_k, log_tau_k, log_w_k

In [ ]:
def physics_informed_loss( p_tau, p_w, p_log_tau, p_log_w, inputs, s_bench):

                                    
    log_t_cc = inputs[:, 0:1] 
    c_cc = inputs[:, 1:2]     
    t_cc = 10 ** log_t_cc     

    s_tensor = s_bench.view(1, -1) 
    y_true = 1.0 / (1.0 + (s_tensor * t_cc) ** c_cc)
    
    s_3d = s_bench.view(1, -1, 1)
    tau_3d = p_tau.unsqueeze(1)
    w_3d = p_w.unsqueeze(1)
    
    denom = 1.0 + s_3d * tau_3d
    terms = w_3d / denom        
    y_pred = torch.sum(terms, dim=-1)
    
    loss_phys = torch.mean(torch.abs((y_true - y_pred))**2)
    
    return loss_phys

In [ ]:
# =================================================================
              
# =================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

n_samples=2000
np.random.seed(42)
torch.manual_seed(42)




tau_samples = np.random.uniform(-4,1, n_samples)
c_samples = np.random.uniform(0.1, 1, n_samples)

X_np = np.stack([tau_samples, c_samples], axis=1)
X_np = np.array(X_np)

                   
dataset = TensorDataset(
    torch.tensor(X_np, dtype=torch.float32).to(device),
    torch.tensor(X_np, dtype=torch.float32).to(device),

)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)


In [ ]:

                                           
freqs_bench = torch.logspace(-2, 6, 100).to(device)
s_bench = 1j * 2 * torch.pi * freqs_bench


In [ ]:
import optuna

# =================================================================
                           
# =================================================================
def objective(trial):
                                                     

    n_poles = trial.suggest_categorical('n_poles', [6, 8, 10, 12, 14, 16])
    width = trial.suggest_categorical('width', [64, 128, 256, 512])
    fourier_size = trial.suggest_categorical('fourier_size', [64, 128, 256, 512])
    num_layers = trial.suggest_categorical('num', [4, 6, 8])
    lr = 1e-4

                            
    model = HighFidelityNeuralVF(
        n_poles=n_poles, 
        width=width, 
        fourier_size=fourier_size,
        num=num_layers
    ).to(device)

                         
    optimizer = optim.Adam(model.parameters(), lr=lr)

                      
    search_epochs = 400

                                                    
    for epoch in range(search_epochs):
        epoch_loss = 0.0

        model.train()
        for batch_x, batch_d0 in dataloader:
            optimizer.zero_grad()
            p_tau, p_w, p_log_tau, p_log_w = model(batch_x)

            loss = physics_informed_loss(
                p_tau, p_w, p_log_tau, p_log_w,
                batch_x, s_bench
            )
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(dataloader)

                                                
        trial.report(avg_loss, epoch)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

                                      
    return avg_loss


In [ ]:
print("[*] 启动贝叶斯超参数调优...")

                                         
                              
study = optuna.create_study(
    direction='minimize', 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=100)                       
)
                
study = optuna.create_study(
    study_name='physics_pinn',                             
    storage='sqlite:///optuna_history.db5',                       
    load_if_exists=True,                     
    direction='minimize', 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=20)
)

      
study.optimize(objective, n_trials=250)

        
print("\n" + "="*50)
print("🎯 调优完成！")
print("="*50)
print(f"最佳验证集 Loss: {study.best_value:.6e}")
print("最佳参数组合:")
for key, value in study.best_params.items():
    print(f"    - {key}: {value}")
print("="*50)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

study = optuna.load_study(
    study_name='physics_pinn', 
    storage='sqlite:///optuna_history.db5'
)

print("历史最佳 Loss:", study.best_value)
print("历史最佳参数:", study.best_params)


In [ ]:

              
df = study.trials_dataframe()

                           
cols_to_keep = ['number', 'value', 'state'] + [col for col in df.columns if col.startswith('params_')]
df_clean = df[cols_to_keep].copy()

           
df_clean.rename(columns={'number': 'Trial', 'value': 'Val_Loss', 'state': 'State'}, inplace=True)

print("\n" + "="*60)
print("📊 所有参数组合搜索结果汇总:")
print("="*60)
                            
print(df_clean.sort_values(by='Val_Loss').to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# =========================
        
# =========================

# =========================
      
# =========================
df_plot = df_clean[df_clean["State"] == "COMPLETE"].copy()
df_plot = df_plot.sort_values("Trial")

param_cols = [col for col in df_plot.columns if str(col).startswith("params_")]

          
best_df = df_plot.nsmallest(1, "Val_Loss")
best_row = best_df.iloc[0]

# =========================
    
# =========================
fig, ax = plt.subplots(figsize=(6, 3))

main_color = "#2f6f9f"        
best_color = "#c0392b"       

               
ax.plot(
    df_plot["Trial"],
    df_plot["Val_Loss"],
    color=main_color,
    linewidth=1.8,
    zorder=2,
    alpha=0.9,
    marker="o",
    markersize=5.2,
    markerfacecolor="white",
    markeredgewidth=1.0,
    markeredgecolor=main_color,
    label="Validation loss",
)

          
ax.scatter(
    best_df["Trial"],
    best_df["Val_Loss"],
    marker="*",
    s=260,
    facecolor=best_color,
    edgecolor="black",
    linewidth=0.7,
    label="Best trial",
    zorder=5,
)

# =========================
         
# =========================
param_text = "\n".join([
    f"{col.replace('params_', '')}: {best_row[col]:.3g}"
    if isinstance(best_row[col], (int, float, np.integer, np.floating))
    else f"{col.replace('params_', '')}: {best_row[col]}"
    for col in param_cols
])

text_str = (
    f"{param_text.capitalize()}"
)

best_x = best_row["Trial"]
best_y = best_row["Val_Loss"]

               
x_min, x_max = df_plot["Trial"].min(), df_plot["Trial"].max()
x_range = x_max - x_min

text_x = best_x + 0.08 * x_range
if text_x > x_max - 0.25 * x_range:
    text_x = best_x - 0.35 * x_range
    ha = "right"
else:
    ha = "left"

text_y = best_y * 3.8

          
ax.annotate(
    text_str,
    xy=(best_x, best_y),
    xytext=(text_x, text_y),
    textcoords="data",
    fontsize=10,
    va="bottom",
    ha=ha,
    linespacing=1.55,
    bbox=dict(
        boxstyle="round,pad=0.35,rounding_size=0.12",
        fc="#f7f7f7",
        ec="#333333",
        lw=0.8,
        alpha=0.96,
    ),
    arrowprops=dict(
        arrowstyle="->",
        color=best_color,
        lw=1.2,
        shrinkA=5,
        shrinkB=5,
        connectionstyle="arc3,rad=0.15",
    ),
    zorder=6,
)

# =========================
       
# =========================
ax.set_xlabel("Trial Number", fontsize=13)
ax.set_ylabel("Validation Loss",  fontsize=13)
ax.set_yscale("log")
ax.set_ylim(1e-5, 5e-4)

ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

    
ax.grid(
    True,
    which="major",
    linestyle="--",
    linewidth=0.65,
    alpha=0.32,
)

ax.grid(
    True,
    which="minor",
    linestyle=":",
    linewidth=0.45,
    alpha=0.18,
)

       
for spine in ax.spines.values():
    spine.set_linewidth(1.2)

ax.tick_params(axis="both", which="major", width=1.0)
ax.tick_params(axis="both", which="minor", width=0.8)

    
ax.legend(
    loc="lower left",
    frameon=False,
    handlelength=1.6,
    borderpad=0.3,
)

plt.tight_layout()
plt.savefig("optuna_trials_curve.png", dpi=400, bbox_inches="tight")

plt.show()